# Local hybrid comment search: quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericksbr/comment_search/blob/main/notebooks/quickstart.ipynb)

This notebook builds a search index over a comment file and runs example searches.
Everything runs in this runtime. Models are downloaded once from Hugging Face, and comment text is never sent to an external service.

Run the cells top to bottom. The first run builds the index (embeddings are cached, so a restart resumes). Later runs load it from the index folder.

## 1. Install (Colab only)

In [ ]:
import os, sys, subprocess, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "commentsearch @ git+https://github.com/ericksbr/comment_search.git"], check=True)

## 2. Mount Google Drive (Colab only)
The data and the index folder live on Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Settings
Change the paths and column names here. Environment variables override the defaults (used for local runs).

In [ ]:
from pathlib import Path

DATA_PATH = os.environ.get(
    "COMMENTS_PATH",
    "/content/drive/MyDrive/claude_code/grouper-topic/data/modified/owrd_div512/grouper/comments_clean_grouped_v2.csv",
)
INDEX_DIR = Path(os.environ.get(
    "COMMENTSEARCH_INDEX_DIR",
    "/content/drive/MyDrive/claude_code/commentsearch/owrd_div512",
))

ID_COL = "part_id"          # unique comment ID
TEXT_COL = "body_clean"     # comment text
DATE_COL = "date"           # parsed as a date; blank or "None" becomes missing
GROUP_COL = "group_id"      # existing Grouper groups, reused as duplicate links (set to None to ignore)
META_COLS = ["commenter_id", "commenter_name", "file", "n_pages", "ocr_suggested", "group_label", "family_id"]

MODEL_NAME = "BAAI/bge-small-en-v1.5"      # dense embeddings (any sentence-transformers model)
RERANKER_NAME = "BAAI/bge-reranker-base"   # optional cross-encoder
TARGET_WORDS, OVERLAP_WORDS = 250, 50
DUP_THRESHOLD = 0.95                        # cosine on mean comment embeddings

print("data:", DATA_PATH)
print("index:", INDEX_DIR)

## 4. Load and clean the comments

In [ ]:
from commentsearch.ingest import load_table, prepare_comments

timings = {}
t = time.perf_counter()
raw = load_table(DATA_PATH)
comments = prepare_comments(raw, ID_COL, TEXT_COL, META_COLS, date_col=DATE_COL,
                            group_col=GROUP_COL if GROUP_COL in raw.columns else None)
timings["ingest"] = round(time.perf_counter() - t, 3)
print(f"{len(raw):,} rows read, {len(comments):,} comments kept, {comments.attrs['dropped_empty']} empty dropped")
print(f"{int(comments['boilerplate'].sum())} flagged as short or 'see attached' (kept in the index)")
comments.head(3)

## 5. Build the index, or load it if it exists
The first build embeds every passage in batches with a progress bar. If the runtime stops partway, rerunning resumes from the cached embeddings.

In [ ]:
from commentsearch import CommentIndex, STEncoder, Searcher

t = time.perf_counter()
encoder = STEncoder(MODEL_NAME)     # downloads the model once from Hugging Face
timings["load_model"] = round(time.perf_counter() - t, 3)

if (INDEX_DIR / "meta.json").exists():
    t = time.perf_counter()
    index = CommentIndex.load(INDEX_DIR)
    timings["load_index"] = round(time.perf_counter() - t, 3)
    print("Loaded existing index:", index.meta["created_utc"])
else:
    index = CommentIndex.build(
        comments, encoder,
        target_words=TARGET_WORDS, overlap_words=OVERLAP_WORDS,
        cache_dir=INDEX_DIR,
        group_col=GROUP_COL if GROUP_COL in comments.columns else None,
        dup_threshold=DUP_THRESHOLD,
        batch_size=32,
    )
    t = time.perf_counter()
    index.save(INDEX_DIR)
    timings["save_index"] = round(time.perf_counter() - t, 3)
    print("Built and saved a new index.")

import pandas as pd
step_times = pd.Series({**timings, **index.timings}, name="seconds")
print(f"{index.meta['n_comments']:,} comments, {index.meta['n_passages']:,} passages, "
      f"duplicate pairs: {index.meta['dedupe_pairs']}")
step_times.to_frame()

## 6. Example searches
Each query runs hybrid search (RRF) with reranking of the top 50 passages. The timings for each step are printed under the results.

In [ ]:
searcher = Searcher(index, encoder=encoder, reranker_name=RERANKER_NAME)

EXAMPLE_QUERIES = [
    "economic impacts on farms and ranches",
    "wildlife habitat and streams",
    "legal authority of the Water Resources Department under state statute",
    "domestic wells going dry from groundwater pumping",
    "managed aquifer recharge and science data",
]

def show(frame, n=5, width=260):
    cols = ["rank", "score", "comment_id", "group_size", "commenter_name", "snippet"]
    out = frame[cols].head(n).copy()
    out["snippet"] = out["snippet"].str.slice(0, width)
    return out

example_results = {}
for q in EXAMPLE_QUERIES:
    frame = searcher.search(q, top_k=10, method="rrf", rerank=True, collapse=True)
    example_results[q] = frame
    print(f"\n=== {q} ===")
    print("  ".join(f"{k} {v:.3f}s" for k, v in frame.attrs["timings"].items()))
    display(show(frame))

## 7. Compare mode
Top 10 for one query under four settings: dense only, BM25 only, hybrid (RRF), and hybrid with reranking.

In [ ]:
COMPARE_QUERY = "economic impacts on farms and ranches"
views = searcher.compare(COMPARE_QUERY, top_k=10)
side = pd.concat(
    {name: frame[["comment_id", "score", "group_size"]].reset_index(drop=True) for name, frame in views.items()},
    axis=1,
)
print(f"Query: {COMPARE_QUERY!r}")
print("Timings (s):", {name: frame.attrs["timings"]["total"] for name, frame in views.items()})
side

## 8. Search UI
Query box, top-k, reranker and collapse toggles, fusion method, metadata filters, and CSV download. The UI runs in this runtime. `share=True` would route traffic through Gradio's servers, so it is off here.

In [ ]:
from commentsearch.app import make_app

demo = make_app(searcher, title="Comment search (prototype)")
# prevent_thread_lock=True returns control to the notebook; without it the cell blocks until the UI is closed.
demo.launch(share=False, prevent_thread_lock=True)